In [1]:
import gradio as gr
import io
import pdfplumber
import requests
from new_functions import (
    extract_resume_text,
    sanitize_input,
    prompt_creator,
    get_resume_response,
    process_resume,
    export_resume,
    cover_letter_prompt_creator,
    get_cover_response,
    save_cover_letter,
    is_posting_recent,
    template_detector,
    mentioned_on_socials,
    source_link_meta_date,
    detect_urgency_language,
    detect_job_board_source,
    career_page_search_url
)
from bs4 import BeautifulSoup 

with gr.Blocks() as app:
    # --- Header ---
    gr.Markdown("""
    <h1 style='text-align:center; color:#1e90ff;'>🥇 Welcome To ResumeWhip!!</h1>
    <h2 style='text-align:center; color:#dd1eff;'>Where We Help You Land Your Next Job with the Help of AI!</h2> 
    <h3 style='text-align:center;'>Powerful Simplicity: Just Verify → Upload → Optimize → Apply!</h3>
    """)

    with gr.Row():
        # --- Sidebar (simplified into Accordions) ---
        with gr.Column(scale=1):
            with gr.Accordion("🦮 How To Use This Website", open = False):
                gr.Markdown("""
                        1.) Crank Your Existing Resume To 11 - list every single skill and experience you have
                            (this is how our AI writes your resume and scores your chances);  
                        2.) Follow the Prompts To Load the Requested Info;  
                        3.) Choose Your Tool (you don't have to use all 3);  
                        4.) Proofread / Edit the Results Using the 'Copy/Pastes' below;  
                        5.) Download Your File as PDF; and 
                        6.) Apply!
                            """)
                
            with gr.Accordion("📚 Copy/Pastes for Resume Formatting", open=False):
                gr.Code("""
If you're not happy with the default resume 
format, you can make adjustments using the 
simple copy and pastes below:
                        
Want To Change Fonts:
# = Biggest  
## = Smaller  
### = Smallest
<b>text</b> = Bold  
<i>text</i> = Italic  
<u>text</u> = Underline
(⬆️ Can Be Combined)
                        
How About A List?
- = Bullet Point  
1. = Numbered List

Link Your Website:
[Your Website](https://www.yourwebsite.com)

Too "clumpy?" Break things up into separate lines, 
by leaving two spaces where you want the line 
to break (e.g. after a period).

Page cuts off where you don't want it to?
Start A New Page (copy/paste entire line below):
<div style="page-break-after: always; break-after: page;"></div>                      
                """, language="markdown")

            gr.Markdown("[📬 Need Help? Have Suggestions?](mailto:support@resumewhip.com)")

            gr.Markdown("### 🛡️ We never store, share, or sell your data. Ever.")

            gr.Markdown("### #️⃣ Know someone who could use this in their job search? Share away!")
            gr.HTML("""
                    <!-- Share Icons -->
    <div style="display:flex; flex-direction:row; gap:15px; justify-content:center; margin-top:10px;">
        <a href="https://www.facebook.com/sharer/sharer.php?u=https://resumewhip.com" target="_blank">
            <img src="https://upload.wikimedia.org/wikipedia/commons/0/05/Facebook_Logo_%282019%29.png" style="width:32px; height:32px;">
        </a>
        <a href="https://x.com/intent/post?url=https://resumewhip.com&text=Check%20out%20this%20awesome%20Resume%20Optimizer!" target="_blank">
            <img src="https://upload.wikimedia.org/wikipedia/commons/c/ce/X_logo_2023.svg" alt="X" style="width:36px; height:36px;">
        </a>
        <a href="https://www.linkedin.com/sharing/share-offsite/?url=https://resumewhip.com" target="_blank">
            <img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" style="width:32px; height:32px;">
        </a>
        <a href="https://www.reddit.com/submit?url=https://resumewhip.com&title=Check%20this%20out!" target="_blank">
             <img src="https://cdn.simpleicons.org/reddit/FF4500" alt="Reddit" style="width:32px; height:32px;">
        </a>
    </div>
</div>
""")
#             gr.Markdown("### 💸 Donations appreciated... only if we've helped:")
#             gr.HTML("""
#                     <div style="text-align:left; display:flex; flex-direction:column; gap:10px; margin-top:15px;">
#         <!-- Support Buttons -->
#         <form action="https://www.paypal.com/donate" method="post" target="_blank">
#             <input type="hidden" name="business" value="YOUR_PAYPAL_EMAIL_OR_ID" />
#             <input type="hidden" name="currency_code" value="USD" />
#             <input type="image" src="https://www.paypalobjects.com/en_US/i/btn/btn_donate_LG.gif" 
#                border="0" name="submit" alt="Donate with PayPal" />
#         </form>
#         <a href="https://www.buymeacoffee.com/yourname" target="_blank">
#         <img src="https://cdn.buymeacoffee.com/buttons/default-orange.png" style="height:40px;width:180px;">
#         </a>
#     </div>
# </div>
# """)

        # --- Main App ---
        with gr.Column(scale=5):
            with gr.Row():
                resume_input = gr.File(label="📝 Upload Your Resume Here")
                company_input = gr.Textbox(label="🏢 Drop In the Company Name", placeholder="e.g., Data Clymer")
                job_input = gr.Textbox(label="🔬 Paste Entire Job Description", lines=8)

            gr.Markdown("<h2 style='text-align:center; color:#ff7f50;'>🧰 Tools In the Toolkit</h2>")

            with gr.Tab("📋 Job Validator"):
                with gr.Row():
                    with gr.Column():
                        resume_input = gr.File(label="Upload Your Resume")
                        job_desc_input = gr.Textbox(label="Paste Job Description", lines=10)
                    with gr.Column():
                        jd_date = gr.Textbox(label="Posting Date (YYYY-MM-DD)")
                        jd_title = gr.Textbox(label="Job Title")
                        company_input = gr.Textbox(label="Company Name")  # added for social check

                validate_btn = gr.Button("✅ Validate Job")
                validation_output = gr.Markdown(label="Job Validation Results")

                def full_job_validator(resume_file, job_description, posting_date, company, job_title):
                    # # --- Existing job validation logic ---
                    # recent = is_posting_recent(posting_date)
                    # template_flag = template_detector(job_description)
                    # urgency_flag = detect_urgency_language(job_description)
                    # social_links = mentioned_on_socials(company, job_title)

                    report = f"### 🕒 Posting Date Check:\n"
                    report += "✅ Job appears to be recent.\n" if recent else "⚠️ Job may be outdated.\n"

                    report += f"\n### 🤖 Template Language:\n"
                    report += "⚠️ Generic/template language detected - could be just harvesting candidates.\n" if template_flag else "✅ Posting looks specific enough to be an actual need.\n"

                    report += f"\n### ⚡ Urgency Language:\n"
                    report += "⚠️ Urgency words detected.\n" if urgency_flag else "✅ No urgency language detected.\n"

                    report += f"\n### 🔍 Social Media Mentions:\n"
                    report += f"- [Search on X](<{social_links['x']}>)\n"
                    report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

                    # --- Optional: Resume processing ---
                    if resume_file is not None:
                        resume_report = process_resume(resume_file, job_description)
                        report += f"\n### 📄 Resume Fit Analysis:\n{resume_report}\n"

                    return report

                validate_btn.click(
                    full_job_validator,
                    # inputs=[resume_input, job_desc_input, jd_date, company_input, jd_title],
                    inputs = [jd_date, jd_title],
                    outputs=validation_output
                )

            # with gr.Tab("Job Validator"):
            #     jd_date = gr.Textbox(label="Posting Date (YYYY-MM-DD)")
            #     jd_title = gr.Textbox(label="Job Title")
            #     jd_validate_btn = gr.Button("✅ Validate Job")
            #     jd_validation_result = gr.Markdown()

            #     def validate_job(posting_date, company, job_title, job_description):
            #         recent = is_posting_recent(posting_date)
            #         template_flag = template_detector(job_description)
            #         urgency_flag = detect_urgency_language(job_description)
            #         social_links = mentioned_on_socials(company, job_title)

            #         report = f"### 🕒 Posting Date Check:\n"
            #         report += "✅ Job appears to be recent.\n" if recent else "⚠️ Job may be outdated.\n"

            #         report += f"\n### 🤖 Template Language:\n"
            #         report += "⚠️ Generic/template language detected - could be just harvesting candidates.\n" if template_flag else "✅ Posting looks specific enough to be an actual need.\n"

            #         report += f"\n### 🔍 Social Media Mentions:\n"
            #         report += f"- [Search on X](<{social_links['x']}>)\n"
            #         report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

            #         return report

            #     jd_validate_btn.click(
            #         fn=validate_job,
            #         inputs=[jd_date, company_input, jd_title, job_input],
            #         outputs=jd_validation_result
            #     )
            
            with gr.Tab("Resume Optimizer"):
                run_resume = gr.Button("🧙 Whip Up Some Resume Magic!")
                resume_md = gr.Markdown()
                resume_edit = gr.Textbox(label="Optimized Resume Above. Make Any Edits In This Box. Or Don't - Up To You.", lines=10)
                suggestions = gr.Markdown(label="Suggestions")
                export_resume_btn = gr.Button("⬇ Download as PDF")
                export_resume_result = gr.File()

            with gr.Tab("Cover Letter Generator"):
                run_cover = gr.Button("📝 Write My Cover Letter")
                cover_output = gr.Textbox(label="Cover Letter", lines=12)
                export_cover_btn = gr.Button("⬇ Download as PDF")
                export_cover_result = gr.File()

            # with gr.Tab("Job Validator"):
            #     jd_date = gr.Textbox(label="Posting Date (YYYY-MM-DD)")
            #     jd_title = gr.Textbox(label="Job Title")
            #     jd_validate_btn = gr.Button("✅ Validate Job")
            #     jd_validation_result = gr.Markdown()

            #     def validate_job(posting_date, company, job_title, job_description):
            #         recent = is_posting_recent(posting_date)
            #         template_flag = template_detector(job_description)
            #         social_links = mentioned_on_socials(company, job_title)

            #         report = f"### 🕒 Posting Date Check:\n"
            #         report += "✅ Job appears to be recent.\n" if recent else "⚠️ Job may be outdated.\n"

            #         report += f"\n### 🤖 Template Language:\n"
            #         report += "⚠️ Generic/template language detected.\n" if template_flag else "✅ Looks specific.\n"

            #         report += f"\n### 🔍 Social Media Mentions:\n"
            #         report += f"- [Search on X](<{social_links['x']}>)\n"
            #         report += f"- [Search on LinkedIn](<{social_links['linkedin']}>)\n"

            #         return report

            #     jd_validate_btn.click(
            #         fn=validate_job,
            #         inputs=[jd_date, company_input, jd_title, job_input],
            #         outputs=jd_validation_result
            #     )

            # Resume events
            run_resume.click(fn=process_resume, inputs=[resume_input, job_input], outputs=[resume_md, resume_edit, suggestions])
            export_resume_btn.click(fn=export_resume, inputs=[resume_edit, company_input], outputs=[export_resume_result])

            # Cover letter events
            def generate_cover_letter(resume_file, job_desc):
                resume_text = ""

                # for pdf uploads
                if resume_file.name.endswith(".pdf"):
                    with pdfplumber.open(resume_file.name) as pdf:
                        for page in pdf.pages:
                            resume_txt += page.extract_text() or ""
                
                # .txt, .md, or other file handling
                else:
                    with open(resume_file.name, "r", encoding="utf-8") as f:
                        resume_txt = f.read()
                
                # build prompt / call
                prompt = cover_letter_prompt_creator(resume_txt, job_desc)
                return get_cover_response(prompt)

            def export_cover_handler(cover_text, company):
                pdf, md = save_cover_letter(cover_text, company)
                return pdf 
                # f"✅ PDF saved at: `{pdf}`" if pdf else "❌ Failed to export."

            run_cover.click(fn=generate_cover_letter, inputs=[resume_input, job_input], outputs=[cover_output])
            export_cover_btn.click(fn=export_cover_handler, inputs=[cover_output, company_input], outputs=[export_cover_result])

    # --- Footer ---
    # gr.Markdown("""
    # <hr>
    # <p style='text-align:center; font-size:1.5em; color:gray;'>
    # 🛡️ Your data is never stored, shared, or sold. Ever.
    # </p>
    # """)

# Launch
app.launch(server_name="0.0.0.0", server_port=8080)

2025-09-01 14:37:20,731 - INFO - OpenAI API key was successfully loaded from your .env file.
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/gradio/utils.py:1024: UserWarning: Expected 5 arguments for function <function full_job_validator at 0x15ff551b0>, received 2.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/gradio/utils.py:1028: UserWarning: Expected at least 5 arguments for function <function full_job_validator at 0x15ff551b0>, received 2.
  warnings.warn(
2025-09-01 14:37:20,879 - INFO - HTTP Request: GET http://localhost:8080/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-09-01 14:37:20,895 - INFO - HTTP Request: HEAD http://localhost:8080/ "HTTP/1.1 200 OK"


* Running on local URL:  http://0.0.0.0:8080

To create a public link, set `share=True` in `launch()`.


2025-09-01 14:37:21,160 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
